In [5]:
import os
import sys
import pandas as pd
from glob import glob
from Bio.PDB import PDBParser

def get_b_factors(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("struct", pdb_file)
    b_factors = {}

    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    res_id = residue.get_id()[1]  # residue number
                    b_factors[res_id] = residue["CA"].get_bfactor()
    return b_factors

# Kyte-Doolittle hydrophobicity scale
KD_SCALE = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

# Max ASA (Wilke scale)
MAX_ASA = {
    'A': 129.0, 'R': 274.0, 'N': 195.0, 'D': 193.0, 'C': 167.0,
    'Q': 225.0, 'E': 223.0, 'G': 104.0, 'H': 224.0, 'I': 197.0,
    'L': 201.0, 'K': 236.0, 'M': 224.0, 'F': 240.0, 'P': 159.0,
    'S': 155.0, 'T': 172.0, 'W': 285.0, 'Y': 263.0, 'V': 174.0
}

EXPOSED_THRESHOLD = 0.25  # Relative ASA threshold for exposure


def get_hydrophobicity(aa):
    return KD_SCALE.get(aa.upper(), 0.0)

def get_max_asa(aa):
    return MAX_ASA.get(aa.upper(), 200.0)


def find_pdb_with_prefix(mutants_dir, fasta_prefix):
    for root, _, files in os.walk(mutants_dir):
        for file in files:
            if file.startswith(fasta_prefix) and file.endswith(".pdb"):
                return os.path.join(root, file)
    return None


def compare_structure_features(disorder_df, pdb_before, pdb_after, dssp_exec, keep_all=False):
    dssp_before = get_dssp_features(pdb_before, dssp_exec)
    dssp_after = get_dssp_features(pdb_after, dssp_exec)

    if keep_all:
        residue_ids = sorted(set(dssp_before.keys()) & set(dssp_after.keys()))
        label_map = dict(zip(disorder_df['SeqNo'], disorder_df['LabelChange']))
    else:
        changed = disorder_df[disorder_df['LabelChange'].isin([-1, 1])]
        residue_ids = changed['SeqNo'].values
        label_map = dict(zip(changed['SeqNo'], changed['LabelChange']))

    changes = []
    for res_id in residue_ids:
        if res_id in dssp_before and res_id in dssp_after:
            aa_before = dssp_before[res_id].get('AA', '?')
            aa_after = dssp_after[res_id].get('AA', '?')
            ss_before = dssp_before[res_id]['SS']
            ss_after = dssp_after[res_id]['SS']
            asa_before = dssp_before[res_id]['ASA']
            asa_after = dssp_after[res_id]['ASA']

            max_asa = get_max_asa(aa_before)
            rASA_before = asa_before / max_asa
            rASA_after = asa_after / max_asa
            rASA_change = rASA_after - rASA_before

            exposure_before = "Exposed" if rASA_before > EXPOSED_THRESHOLD else "Buried"
            exposure_after = "Exposed" if rASA_after > EXPOSED_THRESHOLD else "Buried"
            burial_transition = f"{exposure_before}->{exposure_after}"

            hydro_before = get_hydrophobicity(aa_before)
            hydro_after = get_hydrophobicity(aa_after)
            hydro_change = hydro_after - hydro_before

            ss_transition = f"{ss_before}->{ss_after}" if ss_before != ss_after else "None"

            label_numeric = label_map.get(res_id, 0)
            from_to_map = {-1: ("D", "O"), 1: ("O", "D"), 0: ("-", "-")}
            from_label, to_label = from_to_map.get(label_numeric, ("?", "?"))

            changes.append({
                'Residue': res_id,
                'AA_Before': aa_before,
                'AA_After': aa_after,
                'SS_Before': ss_before,
                'SS_After': ss_after,
                'SS_Transition': ss_transition,
                'ASA_Before': asa_before,
                'ASA_After': asa_after,
                'ASA_Change': asa_after - asa_before,
                'rASA_Before': rASA_before,
                'rASA_After': rASA_after,
                'rASA_Change': rASA_change,
                'Exposure_Before': exposure_before,
                'Exposure_After': exposure_after,
                'Burial_Transition': burial_transition,
                'Hydro_Before': hydro_before,
                'Hydro_After': hydro_after,
                'Hydro_Change': hydro_change,
                'From': from_label,
                'To': to_label
            })

    return pd.DataFrame(changes)


# === Main Loop ===

all_results = []
summary_stats = []
for mut in range(1,5):
    fasta_dir= "/home/mkabir3/Research/27_Dispredict3_colab/MutationRandom/AlphaPDB/fastas/Mutants_"+str(mut)
    for fasta_name in os.listdir(fasta_dir):
        if not fasta_name.endswith(".fasta"):
            continue

        fasta_prefix = "_".join(fasta_name.split("_")[:3]) + "_"
        mutation_id = os.path.splitext(fasta_name)[0]

        mutants_dir = os.path.join(
            "/home/mkabir3/Research/27_Dispredict3_colab/MutationRandom/AlphaPDB/oldResults/output/",
            f"Mutants_{mut}"
        )
        pdb_mut = find_pdb_with_prefix(mutants_dir, fasta_prefix)
        comparison_csv = find_matching_file(comparison_dir, fasta_prefix, "_comparison.csv")

        if not pdb_mut:
            print(f"Missing .pdb for prefix {fasta_prefix} in {mutants_dir}", file=sys.stderr)
            continue
        if not comparison_csv:
            print(f"Missing comparison CSV for prefix {fasta_prefix} in {comparison_dir}", file=sys.stderr)
            continue
        if not os.path.exists(pdb_orig):
            print(f"Missing original PDB: {pdb_orig}", file=sys.stderr)
            continue

        comparison_df = pd.read_csv(comparison_csv)
        result_df = compare_structure_features(
            comparison_df, pdb_orig, pdb_mut, dssp_path, keep_all=keep_all_data
        )
        result_df["MutationID"] = mutation_id
        all_results.append(result_df)

        b_factors_before = get_b_factors(pdb_orig)
        b_factors_after = get_b_factors(pdb_mut)
        bf_before = b_factors_before.get(res_id, None)
        bf_after = b_factors_after.get(res_id, None)
        bf_change = bf_after - bf_before if bf_before is not None and bf_after is not None else None


        # === Per-Mutation Summary ===
        aa_changes = (result_df['AA_Before'] != result_df['AA_After']).sum()
        ss_changes = (result_df['SS_Before'] != result_df['SS_After']).sum()
        total_asa_change = result_df['ASA_Change'].abs().sum()
        # mean_rasa_change = result_df['rASA_Change'].mean()
        mean_rasa_change = result_df['rASA_Change'].abs().mean()
        mean_hydro_change = result_df['Hydro_Change'].mean()
        do_count = ((result_df['From'] == "D") & (result_df['To'] == "O")).sum()
        od_count = ((result_df['From'] == "O") & (result_df['To'] == "D")).sum()
        buried_to_exposed = (result_df['Burial_Transition'] == "Buried->Exposed").sum()
        exposed_to_buried = (result_df['Burial_Transition'] == "Exposed->Buried").sum()

        print(f"\n=== Summary for {mutation_id} ===")
        print(f"Amino Acid changes           : {aa_changes}")
        print(f"Secondary Structure changes  : {ss_changes}")
        print(f"Total ASA change (sum)       : {total_asa_change:.2f}")
        print(f"Average rASA change          : {mean_rasa_change:.3f}")
        print(f"Average Hydrophobicity change: {mean_hydro_change:.3f}")
        print(f"Disorder → Order (D→O)       : {do_count}")
        print(f"Order → Disorder (O→D)       : {od_count}")
        print(f"Buried → Exposed             : {buried_to_exposed}")
        print(f"Exposed → Buried             : {exposed_to_buried}")

        summary_stats.append({
            "MutationID": mutation_id,
            "AA_Changes": aa_changes,
            "SS_Changes": ss_changes,
            "Total_ASA_Change": round(total_asa_change, 2),
            "Avg_rASA_Change": round(mean_rasa_change, 3),
            "Avg_Hydro_Change": round(mean_hydro_change, 3),
            "D_to_O": do_count,
            "O_to_D": od_count,
            "Buried_to_Exposed": buried_to_exposed,
            "Exposed_to_Buried": exposed_to_buried
        })

# === Final Outputs ===
if all_results:
    full_df = pd.concat(all_results, ignore_index=True)
    full_df.to_csv("all_structure_change_summary.csv", index=False)
    print("\n✅ Full residue-level summary saved to all_structure_change_summary.csv")

if summary_stats:
    stats_df = pd.DataFrame(summary_stats)
    stats_df.to_csv("mutation_summary_stats.csv", index=False)
    print("✅ Mutation-level summary saved to mutation_summary_stats.csv")


NameError: name 'find_matching_file' is not defined

In [10]:
import os
import sys
import pandas as pd
from Bio.PDB import PDBParser
from glob import glob

# Required external: DSSP executable and function to extract DSSP features
def get_dssp_features(pdb_file, dssp_exec):
    # Placeholder: Replace with actual DSSP feature extraction logic
    raise NotImplementedError("get_dssp_features() must be implemented or imported.")

def find_matching_file(directory, prefix, suffix):
    for file in os.listdir(directory):
        if file.startswith(prefix) and file.endswith(suffix):
            return os.path.join(directory, file)
    return None

def get_b_factors(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("struct", pdb_file)
    b_factors = {}

    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    res_id = residue.get_id()[1]
                    b_factors[res_id] = residue["CA"].get_bfactor()
    return b_factors

# Kyte-Doolittle hydrophobicity scale
KD_SCALE = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

# Max ASA (Wilke scale)
MAX_ASA = {
    'A': 129.0, 'R': 274.0, 'N': 195.0, 'D': 193.0, 'C': 167.0,
    'Q': 225.0, 'E': 223.0, 'G': 104.0, 'H': 224.0, 'I': 197.0,
    'L': 201.0, 'K': 236.0, 'M': 224.0, 'F': 240.0, 'P': 159.0,
    'S': 155.0, 'T': 172.0, 'W': 285.0, 'Y': 263.0, 'V': 174.0
}

EXPOSED_THRESHOLD = 0.25

def get_hydrophobicity(aa): return KD_SCALE.get(aa.upper(), 0.0)
def get_max_asa(aa): return MAX_ASA.get(aa.upper(), 200.0)

def find_pdb_with_prefix(mutants_dir, fasta_prefix):
    for root, _, files in os.walk(mutants_dir):
        for file in files:
            if file.startswith(fasta_prefix) and file.endswith(".pdb"):
                return os.path.join(root, file)
    return None

def compare_structure_features(disorder_df, pdb_before, pdb_after, dssp_exec, keep_all=False):
    dssp_before = get_dssp_features(pdb_before, dssp_exec)
    dssp_after = get_dssp_features(pdb_after, dssp_exec)

    if keep_all:
        residue_ids = sorted(set(dssp_before.keys()) & set(dssp_after.keys()))
        label_map = dict(zip(disorder_df['SeqNo'], disorder_df['LabelChange']))
    else:
        changed = disorder_df[disorder_df['LabelChange'].isin([-1, 1])]
        residue_ids = changed['SeqNo'].values
        label_map = dict(zip(changed['SeqNo'], changed['LabelChange']))

    changes = []
    for res_id in residue_ids:
        if res_id in dssp_before and res_id in dssp_after:
            aa_before = dssp_before[res_id].get('AA', '?')
            aa_after = dssp_after[res_id].get('AA', '?')
            ss_before = dssp_before[res_id]['SS']
            ss_after = dssp_after[res_id]['SS']
            asa_before = dssp_before[res_id]['ASA']
            asa_after = dssp_after[res_id]['ASA']

            max_asa = get_max_asa(aa_before)
            rASA_before = asa_before / max_asa
            rASA_after = asa_after / max_asa

            exposure_before = "Exposed" if rASA_before > EXPOSED_THRESHOLD else "Buried"
            exposure_after = "Exposed" if rASA_after > EXPOSED_THRESHOLD else "Buried"
            burial_transition = f"{exposure_before}->{exposure_after}"

            hydro_before = get_hydrophobicity(aa_before)
            hydro_after = get_hydrophobicity(aa_after)

            label_numeric = label_map.get(res_id, 0)
            from_to_map = {-1: ("D", "O"), 1: ("O", "D"), 0: ("-", "-")}
            from_label, to_label = from_to_map.get(label_numeric, ("?", "?"))

            changes.append({
                'Residue': res_id,
                'AA_Before': aa_before,
                'AA_After': aa_after,
                'SS_Before': ss_before,
                'SS_After': ss_after,
                'SS_Transition': ss_before + '->' + ss_after if ss_before != ss_after else "None",
                'ASA_Before': asa_before,
                'ASA_After': asa_after,
                'ASA_Change': asa_after - asa_before,
                'rASA_Before': rASA_before,
                'rASA_After': rASA_after,
                'rASA_Change': rASA_after - rASA_before,
                'Exposure_Before': exposure_before,
                'Exposure_After': exposure_after,
                'Burial_Transition': burial_transition,
                'Hydro_Before': hydro_before,
                'Hydro_After': hydro_after,
                'Hydro_Change': hydro_after - hydro_before,
                'From': from_label,
                'To': to_label
            })

    return pd.DataFrame(changes)


def get_dssp_features(pdb_path, dssp_exec):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("X", pdb_path)
    model = structure[0]
    dssp = DSSP(model, pdb_path, dssp=dssp_exec)

    features = {}
    for key in dssp.keys():
        res_id = key[1][1]  # Residue number
        ss = dssp[key][2]   # Secondary structure
        asa = dssp[key][3]  # ASA
        aa = dssp[key][1]   # One-letter amino acid
        features[res_id] = {'SS': ss, 'ASA': asa, 'AA': aa}
    return features

from Bio.PDB import PDBParser, DSSP
import pandas as pd
import os
# === CONFIG ===
# comparison_dir = "/home/mkabir3/Research/27_Dispredict3_colab/MutationRandom/DisorderComparisonResults/Mutants_{mut}_Res"
dssp_path =  "/home/mkabir3/.pyenv/shims/mkdssp"
keep_all_data = False
pdb_orig = "/home/mkabir3/Research/27_Dispredict3_colab/PDBAnalyzer/Main.pdb"

# === Main Loop ===
all_results = []
summary_stats = []

for mut in range(1, 5):
    comparison_dir = f"/home/mkabir3/Research/27_Dispredict3_colab/MutationRandom/DisorderComparisonResults/Mutants_{mut}_Res"
    fasta_dir = f"/home/mkabir3/Research/27_Dispredict3_colab/MutationRandom/AlphaPDB/fastas/Mutants_{mut}"
    mutants_dir = f"/home/mkabir3/Research/27_Dispredict3_colab/MutationRandom/AlphaPDB/output/Mutants_{mut}"

    for fasta_name in os.listdir(fasta_dir):
        if not fasta_name.endswith(".fasta"):
            continue

        fasta_prefix = "_".join(fasta_name.split("_")[:3]) + "_"
        mutation_id = os.path.splitext(fasta_name)[0]
        pdb_mut = find_pdb_with_prefix(mutants_dir, fasta_prefix)
        comparison_csv = find_matching_file(comparison_dir, fasta_prefix, "_comparison.csv")

        if not pdb_mut or not comparison_csv or not os.path.exists(pdb_orig):
            print(f"Skipping {mutation_id}: missing files.", file=sys.stderr)
            continue

        comparison_df = pd.read_csv(comparison_csv)
        result_df = compare_structure_features(comparison_df, pdb_orig, pdb_mut, dssp_path, keep_all=keep_all_data)
        result_df["MutationID"] = mutation_id
        all_results.append(result_df)

        b_factors_before = get_b_factors(pdb_orig)
        b_factors_after = get_b_factors(pdb_mut)

        # === Per-Mutation Summary ===
        aa_changes = (result_df['AA_Before'] != result_df['AA_After']).sum()
        ss_changes = (result_df['SS_Before'] != result_df['SS_After']).sum()
        total_asa_change = result_df['ASA_Change'].abs().sum()
        mean_rasa_change = result_df['rASA_Change'].abs().mean()
        mean_hydro_change = result_df['Hydro_Change'].mean()
        do_count = ((result_df['From'] == "D") & (result_df['To'] == "O")).sum()
        od_count = ((result_df['From'] == "O") & (result_df['To'] == "D")).sum()
        buried_to_exposed = (result_df['Burial_Transition'] == "Buried->Exposed").sum()
        exposed_to_buried = (result_df['Burial_Transition'] == "Exposed->Buried").sum()

        print(f"\n=== Summary for {mutation_id} ===")
        print(f"Amino Acid changes           : {aa_changes}")
        print(f"Secondary Structure changes  : {ss_changes}")
        print(f"Total ASA change (sum)       : {total_asa_change:.2f}")
        print(f"Average rASA change          : {mean_rasa_change:.3f}")
        print(f"Average Hydrophobicity change: {mean_hydro_change:.3f}")
        print(f"Disorder → Order (D→O)       : {do_count}")
        print(f"Order → Disorder (O→D)       : {od_count}")
        print(f"Buried → Exposed             : {buried_to_exposed}")
        print(f"Exposed → Buried             : {exposed_to_buried}")

        summary_stats.append({
            "MutationID": mutation_id,
            "AA_Changes": aa_changes,
            "SS_Changes": ss_changes,
            "Total_ASA_Change": round(total_asa_change, 2),
            "Avg_rASA_Change": round(mean_rasa_change, 3),
            "Avg_Hydro_Change": round(mean_hydro_change, 3),
            "D_to_O": do_count,
            "O_to_D": od_count,
            "Buried_to_Exposed": buried_to_exposed,
            "Exposed_to_Buried": exposed_to_buried
        })

# === Final Outputs ===
if all_results:
    pd.concat(all_results, ignore_index=True).to_csv("all_structure_change_summary.csv", index=False)
    print("\n✅ Full residue-level summary saved to all_structure_change_summary.csv")

if summary_stats:
    pd.DataFrame(summary_stats).to_csv("mutation_summary_stats.csv", index=False)
    print("✅ Mutation-level summary saved to mutation_summary_stats.csv")



=== Summary for Mutant_56_Res1_S255Y ===
Amino Acid changes           : 0
Secondary Structure changes  : 0
Total ASA change (sum)       : 1.24
Average rASA change          : 0.001
Average Hydrophobicity change: 0.000
Disorder → Order (D→O)       : 8
Order → Disorder (O→D)       : 0
Buried → Exposed             : 0
Exposed → Buried             : 0

=== Summary for Mutant_11_Res1_S385F ===
Amino Acid changes           : 0
Secondary Structure changes  : 15
Total ASA change (sum)       : 7.44
Average rASA change          : 0.001
Average Hydrophobicity change: 0.000
Disorder → Order (D→O)       : 59
Order → Disorder (O→D)       : 0
Buried → Exposed             : 0
Exposed → Buried             : 0

=== Summary for Mutant_62_Res1_S293F ===
Amino Acid changes           : 0
Secondary Structure changes  : 24
Total ASA change (sum)       : 11.06
Average rASA change          : 0.001
Average Hydrophobicity change: 0.000
Disorder → Order (D→O)       : 70
Order → Disorder (O→D)       : 0
Buried → Ex